In [4]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU is available: {gpus}")
else:
    print("❌ No GPU found — running on CPU")

TensorFlow version: 2.21.0
✅ GPU is available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [5]:
"""
Deepfake Audio Detection - Full Pipeline (v3)
FYP: Anti-Phishing in the Era of Deepfakes
Author: Ma Ka Yu

Changes from v2:
- Per-folder sampling limits to prevent ASVspoof dominance
- Keeps all Kokoro samples
- Supports WAV, MP3, FLAC
- Saves normalisation stats for accurate single-file inference
"""

import os
import glob
import random
import numpy as np
import librosa
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# CONFIG
DATASET_ROOT    = "/mnt/c/Users/Caillou/PycharmProjects/PythonProject/dataset/audio"
REAL_DIR        = os.path.join(DATASET_ROOT, "real")
FAKE_DIR        = os.path.join(DATASET_ROOT, "fake")

OUTPUT_DIR      = "/mnt/c/Users/Caillou/PycharmProjects/PythonProject"
MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, "deepfake_detector.keras")
NORM_STATS_PATH = os.path.join(OUTPUT_DIR, "norm_stats.npy")
RESULTS_DIR     = os.path.join(OUTPUT_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

SAMPLE_RATE     = 16000
DURATION        = 3.0
N_MELS          = 128
HOP_LENGTH      = 512
N_FFT           = 2048

BATCH_SIZE      = 64
EPOCHS          = 30
LEARNING_RATE   = 1e-4
VALIDATION_SPLIT= 0.15
TEST_SPLIT      = 0.15
RANDOM_SEED     = 42

# Total samples per class (real / fake)
SAMPLES_PER_CLASS = 50000

# Per-folder limits for fake subfolders
# None = use equal share of SAMPLES_PER_CLASS
# Integer = hard cap for that folder
FOLDER_LIMITS = {
    "ASVspoof2019_LA_dev"          : 3000,
    "ASVspoof2019_LA_eval"         : 3000,
    "ASVspoof2019_LA_train"        : 3000,
    "LJSpeech-Kokoro-82M_chunked"  : 8369,  # keep all
}
# Everything not listed gets an equal share of remaining budget

# STEP 1: COLLECT FILE PATHS & LABELS

def get_audio_files(directory):
    """Collect WAV, MP3 and FLAC files recursively."""
    files = []
    for ext in ["*.wav", "*.mp3", "*.flac"]:
        files.extend(glob.glob(os.path.join(directory, "**", ext), recursive=True))
    return files


def collect_files(real_dir: str, fake_dir: str):
    """
    Collect real and fake audio files with smart per-folder sampling.
    Label: 0 = real, 1 = fake
    """
    random.seed(RANDOM_SEED)

    # Real files
    real_files = get_audio_files(real_dir)
    if SAMPLES_PER_CLASS and len(real_files) > SAMPLES_PER_CLASS:
        real_files = random.sample(real_files, SAMPLES_PER_CLASS)
    print(f"[Data] Real files : {len(real_files)}")

    # Fake files
    fake_subdirs = [d for d in os.listdir(fake_dir)
                    if os.path.isdir(os.path.join(fake_dir, d))]
    print(f"[Data] Fake subfolders found: {len(fake_subdirs)}")

    # Calculate budget for folders without a hard limit
    fixed_total  = sum(FOLDER_LIMITS[f] for f in FOLDER_LIMITS if f in fake_subdirs)
    free_folders = [d for d in fake_subdirs if d not in FOLDER_LIMITS]
    remaining_budget = max(0, SAMPLES_PER_CLASS - fixed_total)
    per_free_folder  = remaining_budget // len(free_folders) if free_folders else 0

    fake_files = []
    print("\n[Data] Fake sampling breakdown:")
    for subdir in sorted(fake_subdirs):
        folder_path  = os.path.join(fake_dir, subdir)
        folder_files = get_audio_files(folder_path)

        # Determine limit for this folder
        limit = FOLDER_LIMITS.get(subdir, per_free_folder)
        sampled = random.sample(folder_files, min(limit, len(folder_files)))
        fake_files.extend(sampled)
        print(f"  {subdir:<45} {len(sampled):>6} / {len(folder_files)}")

    print(f"\n[Data] Fake files total : {len(fake_files)}")

    if len(real_files) == 0:
        raise ValueError(f"No real audio files found in {real_dir}")
    if len(fake_files) == 0:
        raise ValueError(f"No fake audio files found in {fake_dir}")

    paths  = real_files + fake_files
    labels = [0] * len(real_files) + [1] * len(fake_files)

    combined = list(zip(paths, labels))
    random.shuffle(combined)
    paths, labels = zip(*combined)
    return list(paths), list(labels)

# STEP 2: FEATURE EXTRACTION

def extract_melspectrogram(filepath: str):
    """Load an audio file and return a log-Mel spectrogram."""
    try:
        audio, sr = librosa.load(filepath, sr=SAMPLE_RATE, mono=True)
    except Exception as e:
        print(f"[WARN] Could not load {filepath}: {e}")
        return None

    target_len = int(SAMPLE_RATE * DURATION)
    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))
    else:
        audio = audio[:target_len]

    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr,
        n_mels=N_MELS,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)
    return log_mel


def build_dataset(paths, labels, desc="Building dataset"):
    """Extract features for every file and return X, y arrays."""
    X, y = [], []
    total = len(paths)
    for i, (path, label) in enumerate(zip(paths, labels)):
        if i % 500 == 0:
            print(f"  [{desc}] {i}/{total} ...")
        feat = extract_melspectrogram(path)
        if feat is not None:
            X.append(feat)
            y.append(label)
    X = np.array(X)
    X = X[..., np.newaxis]
    y = np.array(y)
    return X, y

# STEP 3: NORMALISE

def normalise(X_train, X_val, X_test):
    """Normalise using training set stats and save for inference."""
    mean = float(X_train.mean())
    std  = float(X_train.std() + 1e-8)

    np.save(NORM_STATS_PATH, {"mean": mean, "std": std})
    print(f"[Saved] Norm stats → {NORM_STATS_PATH}  (mean={mean:.4f}, std={std:.4f})")

    return ((X_train - mean) / std,
            (X_val   - mean) / std,
            (X_test  - mean) / std)

# STEP 4: MODEL — CNN + BiLSTM

def build_model(input_shape):
    inp = keras.Input(shape=input_shape)

    # CNN block 1
    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)

    # CNN block 2
    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)

    # CNN block 3
    x = layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)

    # Reshape for LSTM
    freq_bins  = x.shape[1]
    time_steps = x.shape[2]
    channels   = x.shape[3]
    x = layers.Reshape((time_steps, freq_bins * channels))(x)

    # Bidirectional LSTM
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
    x = layers.Bidirectional(layers.LSTM(64))(x)
    x = layers.Dropout(0.3)(x)

    # Dense head
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=["accuracy",
                 keras.metrics.Precision(name="precision"),
                 keras.metrics.Recall(name="recall"),
                 keras.metrics.AUC(name="auc")]
    )
    return model

# STEP 5: TRAIN

def compute_class_weight(y):
    n_real = np.sum(y == 0)
    n_fake = np.sum(y == 1)
    total  = len(y)
    return {0: total / (2 * n_real), 1: total / (2 * n_fake)}


def train_model(model, X_train, y_train, X_val, y_val):
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_auc", patience=5, mode="max", restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6
        ),
        keras.callbacks.ModelCheckpoint(
            MODEL_SAVE_PATH, monitor="val_auc", save_best_only=True, mode="max"
        )
    ]

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        class_weight=compute_class_weight(y_train)
    )
    return history


# STEP 6: EVALUATE

def evaluate_model(model, X_test, y_test):
    print("\n[Evaluate] Running on test set...")
    results = model.evaluate(X_test, y_test, verbose=1)
    metrics = dict(zip(model.metrics_names, results))
    print("\n=== Test Metrics ===")
    for k, v in metrics.items():
        print(f"  {k:12s}: {v:.4f}")

    y_pred_prob = model.predict(X_test).flatten()
    y_pred = (y_pred_prob >= 0.5).astype(int)

    print("\n=== Classification Report ===")
    print(classification_report(y_test, y_pred, target_names=["Real", "Fake"]))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Real", "Fake"],
                yticklabels=["Real", "Fake"])
    plt.title("Confusion Matrix")
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "confusion_matrix.png"), dpi=150)
    plt.close()
    print(f"[Saved] {RESULTS_DIR}/confusion_matrix.png")

    return metrics

# STEP 7: PLOT TRAINING HISTORY

def plot_history(history):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, metric, title in zip(
        axes,
        ["loss", "accuracy", "auc"],
        ["Loss", "Accuracy", "AUC"]
    ):
        ax.plot(history.history[metric],          label="Train")
        ax.plot(history.history[f"val_{metric}"], label="Val")
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "training_history.png"), dpi=150)
    plt.close()
    print(f"[Saved] {RESULTS_DIR}/training_history.png")


# STEP 8: SINGLE-FILE INFERENCE

def predict_single(model, filepath: str,
                   norm_stats_path: str = NORM_STATS_PATH) -> dict:
    """
    Run the detector on one audio file.
    Uses training set normalisation stats for correct scaling.
    """
    feat = extract_melspectrogram(filepath)
    if feat is None:
        return {"error": "Could not load file"}

    X = feat[np.newaxis, ..., np.newaxis]

    if os.path.exists(norm_stats_path):
        stats = np.load(norm_stats_path, allow_pickle=True).item()
        mean, std = stats["mean"], stats["std"]
    else:
        print("[WARN] norm_stats.npy not found — using per-file stats (less accurate)")
        mean = X.mean()
        std  = X.std() + 1e-8

    X = (X - mean) / std

    prob_fake = float(model.predict(X, verbose=0)[0][0])
    label = "FAKE" if prob_fake >= 0.5 else "REAL"

    return {
        "file"            : os.path.basename(filepath),
        "label"           : label,
        "confidence_fake" : f"{prob_fake:.2%}",
        "confidence_real" : f"{1 - prob_fake:.2%}"
    }

# MAIN

def main():
    print("=" * 55)
    print("  Deepfake Audio Detection — Full Pipeline v3")
    print("=" * 55)

    # 1. Collect files
    paths, labels = collect_files(REAL_DIR, FAKE_DIR)

    # 2. Train / val / test split (70 / 15 / 15)
    X_paths, X_test_paths, y_, y_test_ = train_test_split(
        paths, labels, test_size=TEST_SPLIT,
        random_state=RANDOM_SEED, stratify=labels
    )
    X_train_paths, X_val_paths, y_train_, y_val_ = train_test_split(
        X_paths, y_,
        test_size=VALIDATION_SPLIT / (1 - TEST_SPLIT),
        random_state=RANDOM_SEED, stratify=y_
    )
    print(f"\n[Split] Train: {len(X_train_paths)} | Val: {len(X_val_paths)} | Test: {len(X_test_paths)}")

    # 3. Extract features
    print("\n[Features] Extracting Mel spectrograms...")
    X_train, y_train = build_dataset(X_train_paths, y_train_, "Train")
    X_val,   y_val   = build_dataset(X_val_paths,   y_val_,   "Val")
    X_test,  y_test  = build_dataset(X_test_paths,  y_test_,  "Test")

    # 4. Normalise + save stats
    X_train, X_val, X_test = normalise(X_train, X_val, X_test)
    print(f"\n[Shape] X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")

    # 5. Build model
    model = build_model(input_shape=X_train.shape[1:])
    model.summary()

    # 6. Train
    print("\n[Train] Starting training...")
    history = train_model(model, X_train, y_train, X_val, y_val)
    plot_history(history)

    # 7. Evaluate
    evaluate_model(model, X_test, y_test)

    # 8. Example inference
    print("\n[Inference] Example single-file prediction:")
    result = predict_single(model, X_test_paths[0])
    for k, v in result.items():
        print(f"  {k}: {v}")

    print(f"\n[Done] Model saved to      : {MODEL_SAVE_PATH}")
    print(f"[Done] Norm stats saved to : {NORM_STATS_PATH}")
    print(f"[Done] Plots saved to      : {RESULTS_DIR}/")


if __name__ == "__main__":
    main()

  Deepfake Audio Detection — Full Pipeline v3
[Data] Real files : 50000
[Data] Fake subfolders found: 14

[Data] Fake sampling breakdown:
  ASVspoof2019_LA_asv_protocols                      0 / 0
  ASVspoof2019_LA_asv_scores                         0 / 0
  ASVspoof2019_LA_cm_protocols                       0 / 0
  ASVspoof2019_LA_dev                             3000 / 24986
  ASVspoof2019_LA_eval                            3000 / 71933
  ASVspoof2019_LA_train                           3000 / 25380
  LJSpeech-Kokoro-82M_chunked                     8369 / 8369
  ljspeech_full_band_melgan                       3263 / 13100
  ljspeech_hifiGAN                                3263 / 13100
  ljspeech_melgan                                 3263 / 13100
  ljspeech_melgan_large                           3263 / 13100
  ljspeech_multi_band_melgan                      3263 / 13100
  ljspeech_parallel_wavegan                       3263 / 13100
  ljspeech_waveglow                               3263 /

/home/caillou/miniconda3/envs/deepfake-gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  [Train] 500/63146 ...
  [Train] 1000/63146 ...
  [Train] 1500/63146 ...
  [Train] 2000/63146 ...
  [Train] 2500/63146 ...
  [Train] 3000/63146 ...
  [Train] 3500/63146 ...
  [Train] 4000/63146 ...
  [Train] 4500/63146 ...
  [Train] 5000/63146 ...
  [Train] 5500/63146 ...
  [Train] 6000/63146 ...
  [Train] 6500/63146 ...
  [Train] 7000/63146 ...
  [Train] 7500/63146 ...
  [Train] 8000/63146 ...
  [Train] 8500/63146 ...
  [Train] 9000/63146 ...
  [Train] 9500/63146 ...
  [Train] 10000/63146 ...
  [Train] 10500/63146 ...
  [Train] 11000/63146 ...
  [Train] 11500/63146 ...
  [Train] 12000/63146 ...
  [Train] 12500/63146 ...
  [Train] 13000/63146 ...
  [Train] 13500/63146 ...
  [Train] 14000/63146 ...
  [Train] 14500/63146 ...
  [Train] 15000/63146 ...
  [Train] 15500/63146 ...
  [Train] 16000/63146 ...
  [Train] 16500/63146 ...
  [Train] 17000/63146 ...
  [Train] 17500/63146 ...
  [Train] 18000/63146 ...
  [Train] 18500/63146 ...
  [Train] 19000/63146 ...
  [Train] 19500/63146 ...
  [Tra

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 94, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 94, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 94, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 47, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 47, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 47, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 47, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 23, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32, 23, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 23, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 23, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 11, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16, 11, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 11, 2048)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 11, 256)        │     2,229,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

 Trainable params: 2,503,361 (9.55 MB)

 Non-trainable params: 448 (1.75 KB)


[Train] Starting training...
Epoch 1/30


E0000 00:00:1777414870.177296     664 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1777414871.620918     828 cuda_dnn.cc:461] Loaded cuDNN version 92000


987/987 ━━━━━━━━━━━━━━━━━━━━ 106s 94ms/step - accuracy: 0.9425 - auc: 0.9779 - loss: 0.1672 - precision: 0.9152 - recall: 0.9600 - val_accuracy: 0.8141 - val_auc: 0.9907 - val_loss: 0.4307 - val_precision: 0.9933 - val_recall: 0.5869 - learning_rate: 1.0000e-04
Epoch 2/30
987/987 ━━━━━━━━━━━━━━━━━━━━ 78s 79ms/step - accuracy: 0.9803 - auc: 0.9970 - loss: 0.0554 - precision: 0.9681 - recall: 0.9884 - val_accuracy: 0.9107 - val_auc: 0.9899 - val_loss: 0.2153 - val_precision: 0.9765 - val_recall: 0.8193 - learning_rate: 1.0000e-04
Epoch 3/30
987/987 ━━━━━━━━━━━━━━━━━━━━ 78s 79ms/step - accuracy: 0.9881 - auc: 0.9988 - loss: 0.0342 - precision: 0.9814 - recall: 0.9922 - val_accuracy: 0.8455 - val_auc: 0.9833 - val_loss: 0.4300 - val_precision: 0.9925 - val_recall: 0.6583 - learning_rate: 1.0000e-04
Epoch 4/30
987/987 ━━━━━━━━━━━━━━━━━━━━ 77s 78ms/step - accuracy: 0.9908 - auc: 0.9992 - loss: 0.0253 - precision: 0.9861 - recall: 0.9934 - val_accuracy: 0.8897 - val_auc: 0.9900 - val_loss: 0.

In [1]:
import tensorflow as tf
import numpy as np
import librosa
import os

# Config
SAMPLE_RATE     = 16000
DURATION        = 3.0
N_MELS          = 128
HOP_LENGTH      = 512
N_FFT           = 2048
MODEL_SAVE_PATH = "/mnt/c/Users/Caillou/PycharmProjects/PythonProject/deepfake_detector.keras"
NORM_STATS_PATH = "/mnt/c/Users/Caillou/PycharmProjects/PythonProject/norm_stats.npy"

# Load model and norm stats
model = tf.keras.models.load_model(MODEL_SAVE_PATH)
stats = np.load(NORM_STATS_PATH, allow_pickle=True).item()
mean, std = stats["mean"], stats["std"]

def predict_single(filepath):
    audio, sr = librosa.load(filepath, sr=SAMPLE_RATE, mono=True)
    target_len = int(SAMPLE_RATE * DURATION)
    audio = np.pad(audio, (0, max(0, target_len - len(audio))))[:target_len]
    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    X = (log_mel[np.newaxis, ..., np.newaxis] - mean) / std
    prob_fake = float(model.predict(X, verbose=0)[0][0])
    return {"file": os.path.basename(filepath), "label": "FAKE" if prob_fake >= 0.5 else "REAL",
            "confidence_fake": f"{prob_fake:.2%}", "confidence_real": f"{1-prob_fake:.2%}"}

result = predict_single("/mnt/c/Users/Caillou/Documents/Record/test2.wav")
print(result)

I0000 00:00:1777442273.228363   18743 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777442273.376902   18743 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777442275.369933   18743 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1777442277.173156   18743 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries co

{'file': 'test2.wav', 'label': 'REAL', 'confidence_fake': '0.00%', 'confidence_real': '100.00%'}
